In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    BertTokenizer, 
    BertForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

In [ ]:
# Checking GPU availablity
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# If GPU is available, print GPU info
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB")

In [ ]:
# LOAD DATASET (REDUCED SIZE FOR CPU)

from datasets import load_dataset
import numpy as np
from collections import Counter
from datasets import DatasetDict

print("=" * 60)
print("LOADING AG NEWS DATASET (OPTIMIZED FOR CPU)")
print("=" * 60)

# Load full dataset
full_dataset = load_dataset("ag_news")

# Class names
class_names = ["World", "Sports", "Business", "Sci/Tech"]
num_labels = len(class_names)

# === CONFIGURATION - CHANGE THIS ===
TRAIN_SIZE = 1000      # Set your desired size: 6000 = ~5 hours
TEST_SIZE = 500       # Test set size

# Create balanced subset
samples_per_class = TRAIN_SIZE // 4
train_indices = []

for class_id in range(4):
    # Get indices for this class
    class_indices = [i for i, item in enumerate(full_dataset['train']) 
                    if item['label'] == class_id]
    # Randomly select samples_per_class
    selected = np.random.choice(class_indices, min(samples_per_class, len(class_indices)), 
                                replace=False)
    train_indices.extend(selected)

# Shuffle indices
np.random.shuffle(train_indices)

# Create dataset
dataset = DatasetDict({
    'train': full_dataset['train'].select(train_indices),
    'test': full_dataset['test'].select(range(TEST_SIZE))
})

print(f"\n✅ Dataset loaded successfully!")
print(f"Training samples: {len(dataset['train']):,} (reduced from 120,000)")
print(f"Test samples: {len(dataset['test']):,}")
print(f"Number of classes: {num_labels}")
print(f"Class names: {class_names}")

# Verify class distribution
train_labels = [dataset['train'][i]['label'] for i in range(len(dataset['train']))]
print("\n📊 Class distribution in training set:")
for i, name in enumerate(class_names):
    count = Counter(train_labels)[i]
    print(f"  {name:10s}: {count:4d} samples ({count/len(train_labels)*100:.1f}%)")

print(f"\n⏱️  Estimated training time: ~{TRAIN_SIZE * 0.03:.1f} minutes on CPU")
print("=" * 60)

In [ ]:
# Calculate class distribution in training set
train_labels = [dataset['train'][i]['label'] for i in range(len(dataset['train']))]
train_counts = pd.Series(train_labels).value_counts().sort_index()

# Plot class distribution
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
bars = plt.bar(class_names, train_counts.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
plt.title('Class Distribution in Training Set', fontsize=14, fontweight='bold')
plt.xlabel('News Categories', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.xticks(rotation=45)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}', ha='center', va='bottom', fontsize=10)

# Pie chart
plt.subplot(1, 2, 2)
plt.pie(train_counts.values, labels=class_names, autopct='%1.1f%%', 
        colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
plt.title('Class Distribution Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📊 Class Distribution Summary:")
print(f"Average samples per class: {train_counts.mean():.0f}")
print(f"Min samples: {train_counts.min():,} (Class: {class_names[train_counts.argmin()]})")
print(f"Max samples: {train_counts.max():,} (Class: {class_names[train_counts.argmax()]})")

In [ ]:
# REDUCED DATASET SIZE AFTER TOKENIZATION

print("=" * 60)
print("📊 DATASET SIZE AFTER TOKENIZATION")
print("=" * 60)

# Check the size of tokenized datasets
train_size = len(tokenized_datasets['train'])
test_size = len(tokenized_datasets['test'])

print(f"Training samples: {train_size:,}")
print(f"Test samples: {test_size:,}")

if train_size < 10000:
    print(f"\n✅ SUCCESS! Using reduced dataset with {train_size:,} samples")
    print(f"Estimated training time: ~{train_size * 0.03:.1f} minutes")
    
    # Show class distribution in tokenized dataset
    import numpy as np
    from collections import Counter
    
    labels = [tokenized_datasets['train'][i]['labels'].item() for i in range(min(100, train_size))]
    unique_labels = Counter(labels)
    print(f"\n📊 Class distribution (first {min(100, train_size)} samples):")
    for i, name in enumerate(class_names):
        count = unique_labels.get(i, 0)
        print(f"  {name:10s}: {count} samples")
else:
    print(f"\n⚠️ WARNING: Still using large dataset with {train_size:,} samples")
    print("Please check your data loading cell - it should have TRAIN_SIZE=2000")
    print("You may need to restart the kernel and run all cells from the beginning.")

print("=" * 60)

In [ ]:
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

print("=" * 60)
print("DATASET READY")
print("=" * 60)
print(f"Training: {len(train_dataset):,} samples")
print(f"Test: {len(test_dataset):,} samples")

In [ ]:
# Define metrics computation function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'f1_score': f1
    }

print("✅ Metrics function defined successfully!")

In [ ]:
# Load pre-trained Bert model
model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels,
    id2label={i: label for i, label in enumerate(class_names)},
    label2id={label: i for i, label in enumerate(class_names)}
)
model.to(device)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,        # Smaller for CPU
    per_device_eval_batch_size=8,
    num_train_epochs=2,                    # Fewer epochs for speed
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

In [52]:
# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("=" * 60)
print("🚀 INITIALIZING TRAINER")
print("=" * 60)

print(f"\n📊 Dataset Verification:")
print(f"  - Training samples: {len(train_dataset):,}")
print(f"  - Test samples: {len(test_dataset):,}")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")

steps_per_epoch = len(train_dataset) // training_args.per_device_train_batch_size
total_steps = steps_per_epoch * training_args.num_train_epochs

print(f"\n⏱️  Training Steps:")
print(f"  - Steps per epoch: {steps_per_epoch}")
print(f"  - Total steps: {total_steps}")
print(f"  - Estimated time: ~{len(train_dataset) * 0.03:.1f} minutes")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n✅ Trainer initialized with REDUCED dataset!")

🚀 INITIALIZING TRAINER

📊 Dataset Verification:
  - Training samples: 1,000
  - Test samples: 500
  - Batch size: 8

⏱️  Training Steps:
  - Steps per epoch: 125
  - Total steps: 250
  - Estimated time: ~30.0 minutes

✅ Trainer initialized with REDUCED dataset!


In [53]:
# Train the model
print("Starting training...")
print(f"Training on {device}")
print("=" * 50)

train_result = trainer.train()

print("=" * 50)
print("Training completed!")
print(f"Training loss: {train_result.training_loss:.4f}")

Starting training...
Training on cpu


Epoch,Training Loss,Validation Loss,Accuracy,F1 Score
1,0.611431,0.422717,0.884000,0.883111
2,0.299076,0.372691,0.890000,0.890089


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Training completed!
Training loss: 0.5548


In [ ]:
# ============================================
# FIX: Recreate datasets with proper padding
# ============================================

print("=" * 60)
print("🔄 FIXING DATASET FOR EVALUATION")
print("=" * 60)

from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

# Step 1: Ensure all sequences have the same length by using padding
def tokenize_function_fixed(examples):
    return tokenizer(
        examples["text"], 
        padding="max_length",  # Pad to max_length
        truncation=True, 
        max_length=128
    )

# Re-tokenize the datasets with padding
print("Re-tokenizing with fixed padding...")
tokenized_datasets_fixed = dataset.map(tokenize_function_fixed, batched=True)

# Remove text column and rename label
tokenized_datasets_fixed = tokenized_datasets_fixed.remove_columns(["text"])
tokenized_datasets_fixed = tokenized_datasets_fixed.rename_column("label", "labels")

# Set format to torch
tokenized_datasets_fixed.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Create new train/test datasets
train_dataset_fixed = tokenized_datasets_fixed["train"]
test_dataset_fixed = tokenized_datasets_fixed["test"]

print(f"✅ Fixed training samples: {len(train_dataset_fixed)}")
print(f"✅ Fixed test samples: {len(test_dataset_fixed)}")

# Verify shapes are consistent
print("\n📊 Verifying tensor shapes:")
sample = test_dataset_fixed[0]
print(f"Input IDs shape: {sample['input_ids'].shape}")
print(f"Attention mask shape: {sample['attention_mask'].shape}")
print(f"Label: {sample['labels'].item()}")

# Step 2: Now evaluate with fixed dataset
print("\n" + "=" * 60)
print("📊 EVALUATING MODEL ON FIXED TEST SET")
print("=" * 60)

model.eval()

# Create dataloader with fixed dataset
eval_dataloader = DataLoader(test_dataset_fixed, batch_size=16, shuffle=False)

all_predictions = []
all_labels = []
total_loss = 0
num_batches = 0

with torch.no_grad():
    for batch in eval_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        total_loss += outputs.loss.item()
        num_batches += 1
        
        predictions = torch.argmax(outputs.logits, dim=-1)
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
from sklearn.metrics import accuracy_score, f1_score

avg_loss = total_loss / num_batches
accuracy = accuracy_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions, average='weighted')

print("\n✅ EVALUATION COMPLETE!")
print("-" * 40)
print(f"Test Loss: {avg_loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Test F1 Score: {f1:.4f}")
print("-" * 40)

# Save results
import json
import os

evaluation_results = {
    'eval_loss': avg_loss,
    'eval_accuracy': accuracy,
    'eval_f1_score': f1
}

os.makedirs("./results", exist_ok=True)
with open("./results/eval_results.json", "w") as f:
    json.dump(evaluation_results, f, indent=2)
print("📁 Results saved to ./results/eval_results.json")

📊 EVALUATING MODEL ON TEST SET


RuntimeError: stack expects each tensor to be equal size, but got [32] at entry 0 and [77] at entry 1